# How many models for grid search?

How many models will be built when the cross-validator below is fit to data?

<img src="Ex4.jpg" align = "left"> 

## Possible Answers

A) 3  
B) 5  
C) 72  
D) 360

<details><summary>Check your answer</summary><br/>
'''
C)
'''
</details>

Correct! There are 72 points in the parameter grid and 5 folds in the cross-validator. The product is 360. It takes time to build all of those models, which is why you're not doing it here!

In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [3]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
schema = StructType([
    StructField("id", IntegerType()),
	StructField("text", StringType()),
	StructField("label", IntegerType())
	])
	
sms = spark.read.csv("file:///home/talentum/test-jupyter/c5-MLWithPySpark/M4-EnsemblesAndPipelines/3_GridSearch/dataset/sms.csv", 
                     sep=';', header=False, schema=schema)
					 
from pyspark.sql.functions import regexp_replace
sms = sms.withColumn('text', regexp_replace(sms.text, '[_():;,.!?\\\\\\\\-]', ' '))
sms = sms.withColumn('text', regexp_replace(sms.text, '[0-9]', ' '))
sms = sms.withColumn('text', regexp_replace(sms.text, '(^ +| +$)', ''))
sms = sms.withColumn('text', regexp_replace(sms.text, ' +', ' '))

from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
tokenizer = Tokenizer(inputCol='text', outputCol='words')
remover = StopWordsRemover(inputCol=tokenizer.getOutputCol(), outputCol='terms')
hasher = HashingTF(inputCol=remover.getOutputCol(), outputCol="hash")
idf = IDF(inputCol=hasher.getOutputCol(), outputCol="features")
logistic = LogisticRegression()
pipeline = Pipeline(stages = [tokenizer, remover, hasher, idf, logistic])

from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [4]:
# Create parameter grid
params = ParamGridBuilder()

# Add grid for hashing trick parameters
params = params.addGrid(hasher.numFeatures, [1024, 4096, 16384]) \
               .addGrid(hasher.binary, [True, False])

# Add grid for logistic regression parameters
params = params.addGrid(logistic.regParam, [0.01, 0.1, 1.0, 10.0]) \
               .addGrid(logistic.elasticNetParam, [0.0, 0.5, 1.0])

# Build parameter grid
params = params.build()

In [5]:
print ('Number of models to be tested: ', len(params))


Number of models to be tested:  72
